# Automated SME Credit Assessment Pipeline
**Client:** Stanbic Bank Ghana

**Objective:** Prototype an automated credit decisioning pipeline for SME loans to recommend APPROVE, DECLINE, or REFER TO HUMAN.

## Phase 1: Data Ingestion & Ethical Audit
In this initial phase, will load the historical loan application data, inspect the structural integrity of the dataset, and conduct an immediate **Ethical Audit**. To ensure compliance with the Bank of Ghana's fair lending standards, we must proactively identify and isolate any protected classes or sensitive demographic features (e.g., gender, age, geographic origin) before they contaminate our predictive modeling pipeline.

In [10]:
# Import libraries
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, auc
import matplotlib.pyplot as plt 
import joblib
import os

# Suppress warnings for a cleaner notebook presentation
warnings.filterwarnings('ignore')

# Ingest the dataset
file_path = 'ds-sme_loan_applications_stanbic_gh.csv'
df = pd.read_csv(file_path)

# Initial Data Inspection
print("--- Data Shape ---")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}\n")

print("--- Column Names & Data Types ---")
print(df.dtypes)
print("\n--- Missing Values Count ---")
print(df.isnull().sum()[df.isnull().sum() > 0]) 

df.head(3)

--- Data Shape ---
Rows: 3037, Columns: 29

--- Column Names & Data Types ---
application_id                   object
business_name                    object
sector                           object
region                           object
ethnic_group                     object
years_in_operation              float64
owner_gender                     object
owner_age                         int64
disability_status                object
num_employees                     int64
annual_revenue_ghs               object
monthly_momo_volume_ghs         float64
avg_monthly_bank_balance_ghs    float64
bank_account_tenure_months        int64
has_momo_account                 object
gra_tin                          object
loan_amount_requested_ghs       float64
loan_purpose                     object
collateral_type                  object
collateral_value_ghs            float64
previous_loan_count               int64
previous_default                 object
credit_bureau_score             float64
rm

,application_id,business_name,sector,region,ethnic_group,years_in_operation,owner_gender,owner_age,disability_status,num_employees,...,collateral_value_ghs,previous_loan_count,previous_default,credit_bureau_score,rm_recommendation,internal_risk_grade,application_date,contact_phone,days_past_due_current,loan_default
0,SBG-2025-01722,Achimota Farms,Fishing,Ashanti,Akan,0.6,Female,34,No,2,...,29174.78,0,NaN,NaN,Approve,A,05/07/2024,558707981,0,0
1,SBG-2025-02335,Kofi MotorsCo,Hospitality,Savannah,Mole-Dagbani,0.6,Female,31,No,19,...,15816.11,2,Yes,628.0,Refer,B,06/07/2025,270660120,0,0
2,SBG-2025-00444,Dansoman TradingLtd,Hospitality,Greater Accra,Ga-Adangbe,2.3,female,32,No,8,...,13374.67,1,No,371.0,Decline,C,24/10/2025,546332387,0,0


## Phase 2: The Ethical Audit & Deep Data Cleaning
Based on the initial inspection, we must address critical enterprise-level issues:

1. **Ethical Safeguards (Bank of Ghana Compliance):** The dataset contains ethnic_group, owner_gender, owner_age, and disability_status. Including these violates fair lending practices and risks algorithmic bias. We will explicitly drop them.

2. **Target Leakage:** The column days_past_due_current represents the current state of the loan. Including this to predict loan_default for *new* applications is target leakage.

3. **Data Quality & Typing:** annual_revenue_ghs is incorrectly typed as an object. Furthermore, binary categorical variables (like has_momo_account) contain severe input inconsistencies (eg. yes, y, true, 1).

4. **Standardization:** We will wrap our cleaning and mapping logic into a single clean_raw_data function. This satisfies the brief's requirement for a reusable inference module that guarantees new applications are processed identically to the training data.

In [ ]:
# columns to drop for ethical, identification, and leakage reasons
ethical_cols = ['ethnic_group', 'owner_gender', 'owner_age', 'disability_status']
id_cols = ['application_id', 'business_name', 'contact_phone', 'gra_tin', 'application_date']
leakage_cols = ['days_past_due_current']

cols_to_drop = ethical_cols + id_cols + leakage_cols

def clean_raw_data(data):
    """
    Reusable preprocessing function to clean raw SME application data.
    Designed to handle both bulk training data and single inference requests.
    """
    df_clean = data.copy()
    
    # Drop restricted, identifier, and leaky columns
    df_clean = df_clean.drop(columns=[col for col in cols_to_drop if col in df_clean.columns])
    
    # Fix data types: 'annual_revenue_ghs' object to float
    if 'annual_revenue_ghs' in df_clean.columns and df_clean['annual_revenue_ghs'].dtype == 'object':
        df_clean['annual_revenue_ghs'] = df_clean['annual_revenue_ghs'].astype(str).str.replace(r'[^\d.]', '', regex=True)
        df_clean['annual_revenue_ghs'] = pd.to_numeric(df_clean['annual_revenue_ghs'], errors='coerce')
        
    # Standardize text casing
    categorical_cols = df_clean.select_dtypes(include=['object']).columns
    for col in categorical_cols:
        df_clean[col] = df_clean[col].astype(str).str.strip().str.lower()
        df_clean[col] = df_clean[col].replace('nan', np.nan)

    # Normalize Binary/Boolean Inconsistencies - Maps various forms of yes/no to a standard string
    binary_mapping = {
        'yes': 'yes', 'y': 'yes', 'true': 'yes', '1': 'yes', '1.0': 'yes',
        'no': 'no', 'n': 'no', 'false': 'no', '0': 'no', '0.0': 'no'
    }
    
    # Apply mapping to known binary columns if they exist
    for bin_col in ['has_momo_account', 'previous_default']:
        if bin_col in df_clean.columns:
            # Map valid entries, keep NaNs as NaNs
            df_clean[bin_col] = df_clean[bin_col].map(binary_mapping).fillna(df_clean[bin_col])
            
    return df_clean

# Apply the updated cleaning function
df_cleaned = clean_raw_data(df)

# Separate Features (X) from Target (y)
X = df_cleaned.drop(columns=['loan_default'])
y = df_cleaned['loan_default']

# Verify the fix!
print("--- Normalized Binary Columns Verification ---")
print("has_momo_account unique values:", X['has_momo_account'].unique())
print("previous_default unique values:", X['previous_default'].unique())

# Comprehensive Information Summary
print("=== CLEANED DATA INFO ===")
df_cleaned.info()

print("\n=== MISSING VALUES COUNT (POST-CLEANING) ===")
# Only show columns that actually have missing values
missing_counts = df_cleaned.isnull().sum()
print(missing_counts[missing_counts > 0].sort_values(ascending=False))

print("\n=== SUMMARY STATISTICS (NUMERICAL) ===")
# lambda function to format the output to avoid scientific notation
display(df_cleaned.describe().apply(lambda s: s.apply('{0:.2f}'.format)))

--- Normalized Binary Columns Verification ---
has_momo_account unique values: ['yes' 'no']
previous_default unique values: [nan 'yes' 'no']
=== CLEANED DATA INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3037 entries, 0 to 3036
Data columns (total 19 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   sector                        3037 non-null   object 
 1   region                        3037 non-null   object 
 2   years_in_operation            3037 non-null   float64
 3   num_employees                 3037 non-null   int64  
 4   annual_revenue_ghs            2950 non-null   float64
 5   monthly_momo_volume_ghs       2583 non-null   float64
 6   avg_monthly_bank_balance_ghs  2728 non-null   float64
 7   bank_account_tenure_months    3037 non-null   int64  
 8   has_momo_account              3037 non-null   object 
 9   loan_amount_requested_ghs     3037 non-null   float64
 10  loan_purpose 

,years_in_operation,num_employees,annual_revenue_ghs,monthly_momo_volume_ghs,avg_monthly_bank_balance_ghs,bank_account_tenure_months,loan_amount_requested_ghs,collateral_value_ghs,previous_loan_count,credit_bureau_score,loan_default
count,3037.00,3037.00,2950.00,2583.00,2728.00,3037.00,3037.00,2648.00,3037.00,1070.00,3037.00
mean,5.30,11.97,155474.46,6533.98,27144.71,36.25,35176.92,51955.63,1.49,544.21,0.13
std,8.73,16.97,368545.29,11667.64,138322.22,36.32,43715.74,91418.04,1.21,102.82,0.34
min,-2.00,1.00,102.61,55.40,34.72,0.00,569.59,0.00,0.00,103.00,0.00
25%,1.40,3.00,18879.62,1350.48,3023.95,10.00,10865.83,0.00,1.00,475.00,0.00
50%,3.50,7.00,53475.38,3084.00,8370.12,25.00,21418.62,22141.24,1.00,546.00,0.00
75%,7.10,14.00,148392.63,6962.18,23718.37,50.00,43095.39,60115.96,2.00,613.00,0.00
max,200.00,251.00,9498575.16,182534.15,6706758.96,240.00,815331.50,1133168.42,6.00,876.00,1.00


## Phase 3: Outlier Handling, Train-Test Split & The Preprocessing Pipeline
Following our data health check, we identified impossible values in years_in_operation (negative years and extreme 200-year outliers). We apply a hard clip to bound this feature to realistic SME timelines (0 to 50 years). 

With a default rate of 13.3%, we are dealing with a heavily imbalanced dataset. We use a **stratified** train-test split to guarantee the minority class is equally represented in both sets. 

Next, we construct the scikit-learn ColumnTransformer and Pipeline. This fulfills the brief's requirement for reusable inference:
* **Numerical Features:** We impute missing values using the median (robust to financial outliers). Crucially, we use add_indicator=True. Since ~65% of credit_bureau_score values are missing, the indicator explicitly tells the model "this applicant has no credit history," which is a highly predictive signal in itself. We then apply StandardScaler.

* **Categorical Features:** We impute missing values with 'missing' and apply OneHotEncoder, setting handle_unknown=ignore so the pipeline doesn't crash if a new business sector appears during future inference.

In [ ]:
# Outlier Capping (Based on health check findings) - cap negative years at 0, and extreme typos at 50 years
X['years_in_operation'] = X['years_in_operation'].clip(lower=0, upper=50)

# Stratified Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y # Crucial for the 13.3% imbalance
)

# Identify numerical and categorical columns dynamically
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()

# Build the preprocessing pipelines
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine into a ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ])

# Fit the preprocessor STRICTLY on the training data, then transform both
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Extract feature names
cat_feature_names = preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(cat_cols)

num_imputer = preprocessor.named_transformers_['num'].named_steps['imputer']
if hasattr(num_imputer, 'indicator_') and num_imputer.indicator_ is not None:
    # indicator_.features_ returns the precise indices of columns with missing data
    num_indicator_names = [f"{num_cols[i]}_is_missing" for i in num_imputer.indicator_.features_]
else:
    num_indicator_names = []
    
all_feature_names = num_cols + num_indicator_names + list(cat_feature_names)

# Output results
print("--- Pipeline Creation Successful ---")
print(f"Training Features Shape: {X_train_processed.shape}")
print(f"Testing Features Shape: {X_test_processed.shape}")
print(f"Total Feature Space: {len(all_feature_names)} features (including dummies & indicators)")

# Verify stratification worked
print("\n--- Train vs Test Default Rate ---")
print(f"Train Default Rate: {y_train.mean():.3f}")
print(f"Test Default Rate: {y_test.mean():.3f}")

--- Pipeline Creation Successful ---
Training Features Shape: (2429, 72)
Testing Features Shape: (608, 72)
Total Feature Space: 72 features (including dummies & indicators)

--- Train vs Test Default Rate ---
Train Default Rate: 0.133
Test Default Rate: 0.133


## Phase 4: Model Training & Cost-Sensitive Evaluation
For our credit scoring engine, we employ a dual-model challenger/champion strategy:
1. **Logistic Regression (Baseline):** Highly interpretable, linearly separable baseline. Critical for regulatory explanations to the Bank of Ghana.
2. **XGBoost (Challenger):** State-of-the-art gradient boosting that handles non-linear feature interactions natively.

**Addressing Class Imbalance (13.3% Default Rate):** 
Rather than injecting synthetic data via SMOTE (which can warp the statistical reality of financial datasets), we utilize algorithmic class weighting (class_weight= balanced for LR, scale_pos_weight for XGBoost). This mathematically penalizes the model heavier for missing a true Default.

**Evaluation Strategy:**
We explicitly reject Accuracy. A naive model predicting "No Default" achieves 86.7% accuracy but causes catastrophic financial loss. Instead, we optimize for **PR-AUC (Precision-Recall Area Under Curve)** and **F1-Score (Macro)**, focusing the model's performance specifically on the minority class.

In [ ]:
# Initialize Models with Class Weights
# Logistic Regression: 'balanced' automatically adjusts weights inversely proportional to class frequencies
log_reg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

# XGBoost: Calculate scale_pos_weight (ratio of negative to positive instances)
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
xgb_model = xgb.XGBClassifier(
    scale_pos_weight=pos_weight, 
    random_state=42, 
    eval_metric='logloss',
    use_label_encoder=False
)

# Train the Models
print("Training Logistic Regression...")
log_reg.fit(X_train_processed, y_train)

print("Training XGBoost...")
xgb_model.fit(X_train_processed, y_train)

# Generate Predictions (Probabilities and Hard Classes)
# need probabilities for custom Business Engine in the next phase
lr_probs = log_reg.predict_proba(X_test_processed)[:, 1]
xgb_probs = xgb_model.predict_proba(X_test_processed)[:, 1]

lr_preds = log_reg.predict(X_test_processed)
xgb_preds = xgb_model.predict(X_test_processed)

# Evaluation Helper Function
def evaluate_model(model_name, y_true, y_pred, y_probs):
    print(f"\n{'='*40}")
    print(f" MODEL: {model_name} ")
    print(f"{'='*40}")
    
    # Calculate PR-AUC
    precision, recall, _ = precision_recall_curve(y_true, y_probs)
    pr_auc = auc(recall, precision)
    
    print(f"PR-AUC Score: {pr_auc:.4f}\n")
    print("--- Classification Report ---")
    print(classification_report(y_true, y_pred, target_names=['0: Paid', '1: Default']))
    
    print("--- Confusion Matrix ---")
    cm = confusion_matrix(y_true, y_pred)
    print(f"True Negatives (Correct Approvals) : {cm[0][0]}")
    print(f"False Positives (Reckless Approvals): {cm[0][1]}")
    print(f"False Negatives (Missed Defaults)  : {cm[1][0]}")
    print(f"True Positives (Correct Declines)  : {cm[1][1]}")

# Execute Evaluation
evaluate_model("Logistic Regression (Baseline)", y_test, lr_preds, lr_probs)
evaluate_model("XGBoost (Challenger)", y_test, xgb_preds, xgb_probs)

Training Logistic Regression...
Training XGBoost...

 MODEL: Logistic Regression (Baseline) 
PR-AUC Score: 0.9549

--- Classification Report ---
              precision    recall  f1-score   support

     0: Paid       0.99      0.97      0.98       527
  1: Default       0.81      0.91      0.86        81

    accuracy                           0.96       608
   macro avg       0.90      0.94      0.92       608
weighted avg       0.96      0.96      0.96       608

--- Confusion Matrix ---
True Negatives (Correct Approvals) : 510
False Positives (Reckless Approvals): 17
False Negatives (Missed Defaults)  : 7
True Positives (Correct Declines)  : 74

 MODEL: XGBoost (Challenger) 
PR-AUC Score: 0.9701

--- Classification Report ---
              precision    recall  f1-score   support

     0: Paid       0.98      0.99      0.99       527
  1: Default       0.92      0.90      0.91        81

    accuracy                           0.98       608
   macro avg       0.95      0.94      0.

## Phase 5: The Business Decision Engine
A machine learning probability is only as good as the business rules wrapped around it. Here, we build the core decisioning logic required by Stanbic Bank.

1. **Probability Thresholds:** 
   * Default Probability <= 15%: **APPROVE** (Low Risk)
   * Default Probability >= 40%: **DECLINE** (High Risk)
   * Between 15% and 40%: **REFER TO HUMAN** (Grey Area requiring manual underwriting)
   
2. **The Hard Business Override:**
   * Regardless of how confident the model is, if a business has been operating for less than 1 year (years_in_operation < 1.0), the system will automatically strip the "APPROVE" status and force a **REFER TO HUMAN**. Startups lack sufficient macroeconomic resilience, and banking policy requires human eyes on these profiles before capital deployment.

In [ ]:
# a copy of unscaled test set to easily view the original business data
business_df = X_test.copy()

# champion model's predicted probabilities
business_df['predicted_default_prob'] = xgb_probs

# Define the Decision Engine Function
def execute_decision_engine(row):
    prob = row['predicted_default_prob']
    years = row['years_in_operation']
    
    # A: Model-Driven Logic
    if prob >= 0.40:
        decision = 'DECLINE'
    elif prob <= 0.15:
        decision = 'APPROVE'
    else:
        decision = 'REFER TO HUMAN'
        
    # B: Hard Business Override (The Startup Rule)
    # If the model approves, but the business is < 1 year old, force a referral.
    if decision == 'APPROVE' and years < 1.0:
        decision = 'REFER TO HUMAN (Startup Override)'
        
    return decision

# Apply engine to generate final recommendations
business_df['final_recommendation'] = business_df.apply(execute_decision_engine, axis=1)

# Output the final distribution as required by the brief
print("--- Final Business Recommendations Distribution ---")
recommendation_counts = business_df['final_recommendation'].value_counts()
print(recommendation_counts)

print("\n--- Sanity Check: Overrides in Action ---")
# Show examples where the model wanted to approve, but the hard rule intervened
overrides = business_df[business_df['final_recommendation'] == 'REFER TO HUMAN (Startup Override)']
if not overrides.empty:
    display(overrides[['years_in_operation', 'predicted_default_prob', 'final_recommendation']].head(3))
else:
    print("No startup overrides triggered in this sample.")

--- Final Business Recommendations Distribution ---
final_recommendation
APPROVE                              427
REFER TO HUMAN (Startup Override)     93
DECLINE                               79
REFER TO HUMAN                         9
Name: count, dtype: int64

--- Sanity Check: Overrides in Action ---


,years_in_operation,predicted_default_prob,final_recommendation
2046,0.1,0.001332,REFER TO HUMAN (Startup Override)
2380,0.3,0.000162,REFER TO HUMAN (Startup Override)
1310,0.5,0.000506,REFER TO HUMAN (Startup Override)


## Phase 6: Model Serialization & Pipeline Export
To deploy this prototype into a production environment, we must serialize both the trained XGBoost model and the scikit-learn preprocessing pipeline. 

Crucially, we export the ColumnTransformer (preprocessor) alongside the model. This guarantees **Training-Serving Skew prevention**. When a new SME application arrives in production, it will be imputed, scaled, and encoded using the exact same mathematical parameters learned during this training phase. We use joblib for serialization as it is highly optimized for large numpy arrays common in scikit-learn objects.

In [12]:
# Define the export directory (saves in the current working directory)
export_dir = './'

# Serialize the Preprocessing Pipeline
pipeline_path = os.path.join(export_dir, 'sme_preprocessor.pkl')
joblib.dump(preprocessor, pipeline_path)
print(f" Preprocessor successfully saved to: {pipeline_path}")

# Serialize the Champion Model (XGBoost)
model_path = os.path.join(export_dir, 'sme_xgboost_model.pkl')
joblib.dump(xgb_model, model_path)
print(f" Champion Model successfully saved to: {model_path}")

# Final Sanity Check: Verify files exist and print their sizes
print("\n--- File Verification ---")
for file in ['sme_preprocessor.pkl', 'sme_xgboost_model.pkl']:
    file_stat = os.stat(file)
    # Convert bytes to Kilobytes for readability
    print(f"{file}: {file_stat.st_size / 1024:.2f} KB")

print("\n NOTEBOOK PIPELINE COMPLETE. READY FOR STREAMLIT DEPLOYMENT.")

 Preprocessor successfully saved to: ./sme_preprocessor.pkl
 Champion Model successfully saved to: ./sme_xgboost_model.pkl

--- File Verification ---
sme_preprocessor.pkl: 7.81 KB
sme_xgboost_model.pkl: 127.97 KB

 NOTEBOOK PIPELINE COMPLETE. READY FOR STREAMLIT DEPLOYMENT.
